# Faster R-CNN — ablation seed42 (một đối chứng mỗi phiên)

Năm lựa chọn: `lr`, `weight_decay`, `dropout`, `jitter`, `standard`.
Mỗi bản Kaggle chọn một mục ở cell settings, chạy độc lập (song song nếu tài khoản cho phép).
Ba mục đầu trả tham số về baseline; jitter tắt affine jitter; standard thay kiến trúc head.
Đây là các đối chứng có điều kiện trong recipe winner, không phải tuning riêng standard.
Mỗi phiên dùng T4×2 và tạo một run Forge riêng trong sandbox; có thể chạy song song ở các phiên độc lập.
Add Input: cache đã dùng + Output tuning cuối chứa bundle e2d80e82.zip và checksum.
Bật Internet, GPU T4×2, Secret WANDB_API_KEY. Save & Run All thực sự train.
Giữ source training a99d294, cap100/warmup3/ES10/batch24/eval32/seed42.
Train mới từ COCO pretrained; không resume/checkpoint winner, không tạo trial9.
Sau mỗi phiên tải ZIP+SHA256 ở Output gốc; không cần upload repo/notebook mới để source pin chạy được.


In [ ]:
ABLATION = "lr"  # lr / weight_decay / dropout / jitter / standard; mỗi phiên chọn một mục
REFERENCE_BUNDLE = ""  # Để trống nếu chỉ có đúng 1 bundle e2d80e82 ở Input; hoặc Copy file path
RESTORE_OUTPUT = ""  # Chỉ điền ZIP ablation của chính lựa chọn này khi khôi phục phiên cũ
START_TRAIN = True
assert ABLATION in {"lr", "weight_decay", "dropout", "jitter", "standard"}


In [ ]:
import os, sys, subprocess, json, re
from pathlib import Path

REPO_URL = "https://github.com/tuan8p/VN-Traffic-Sign-Detection.git"
REPO_REF = "a99d2946b55305cea8c6049bd6f581ab709f876a"  # SHA đầy đủ của commit có tích hợp Sweeps, đã push; không dùng tên branch.
assert re.fullmatch(r"[0-9a-f]{40}", REPO_REF), "Điền SHA đã commit/push trước khi chạy."
assert Path("/kaggle/working").exists(), "Notebook này dành cho Kaggle."
REPO = Path("/kaggle/working/VN-Traffic-Sign-Detection")

def run(command, **kwargs):
    return subprocess.run([str(x) for x in command], check=True, **kwargs)

if not (REPO / ".git").exists():
    run(["git", "clone", "--no-checkout", REPO_URL, REPO])
else:
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True)
    assert not dirty.strip(), "Repo có tracked changes; giữ lại trước khi checkout."
run(["git", "-C", REPO, "fetch", "origin", REPO_REF])
run(["git", "-C", REPO, "checkout", "--detach", REPO_REF])
os.chdir(REPO)
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[detect]", "wandb==0.31.0"])
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Source:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
import torch, torchvision, wandb
from torchvision.ops import nms
from kaggle_secrets import UserSecretsClient
from vn_tsd.utils.io import load_yaml, save_yaml

shared = load_yaml(REPO / "configs/shared.yaml")
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_BASE_URL"] = shared["project"]["wandb_base_url"]
os.environ["WANDB_MODE"] = "online"
print("torch", torch.__version__, "torchvision", torchvision.__version__, "wandb", wandb.__version__)
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Cần đúng hai GPU T4."
for index in range(2):
    device = f"cuda:{index}"
    result = nms(torch.tensor([[0., 0., 10., 10.]], device=device), torch.tensor([.9], device=device), .5)
    assert result.tolist() == [0]
    print(index, torch.cuda.get_device_name(index))
for key in ("WANDB_SWEEP_ID", "WANDB_RUN_ID", "WANDB_CONFIG_PATHS", "WANDB_RESUME"):
    os.environ.pop(key, None)  # Notebook này không thuộc campaign tuning cũ.


## Kiểm nguồn tham chiếu và tạo đối chứng (chưa train)

Chỉ đọc JSON/YAML từ bundle, không extract checkpoint hoặc load torch weights.
Checksum được pin theo bundle đã kiểm local, không chỉ tin sidecar upload.
Source/runtime/cache phải khớp campaign tuning; khác thì dừng để review.


In [ ]:
import copy, hashlib, math, stat, uuid
from zipfile import ZipFile, ZIP_STORED
from vn_tsd.pipelines.faster_rcnn.sweep import source_identity, runtime_identity
from vn_tsd.pipelines.faster_rcnn.dataset import content_hash, file_hash, load_cache
from vn_tsd.pipelines.faster_rcnn.config import validate_config
from vn_tsd.pipelines.faster_rcnn.artifacts import write_json

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")
REFERENCE_NAME = "frcnn_tune_s42_es10_bayes8_sandbox_v1-artifacts-e2d80e82.zip"
REFERENCE_SHA256 = "3c817755f338f50e5300fc5e9b5b2b3604969a2c6bf58a384f714036f46f394f"
WINNER_ID = "ahgf5kge"
REFERENCE_CAMPAIGN = "frcnn_tune_s42_es10_bayes8_sandbox_v1"
CHOICES = {"lr": ("train", "lr", 0.002),
           "weight_decay": ("train", "weight_decay", 0.0005),
           "dropout": ("model", "head_dropout", 0.2),
           "jitter": ("online_preprocess", "jitter", 0.0),
           "standard": ("model", "head", "standard")}
EXPECTED_WINNER = {"train.lr": 0.009606472124890571,
                   "train.weight_decay": 1.4756984677083597e-05,
                   "model.head_dropout": 0.4, "online_preprocess.jitter": 0.1}

def read_reference(bundle):
    bundle = Path(bundle)
    checksum = bundle.with_suffix(".zip.sha256")
    assert checksum.is_file(), f"Thiếu checksum: {checksum}"
    actual = file_hash(bundle)
    assert actual == REFERENCE_SHA256 == checksum.read_text().split()[0], "Sai bundle/checksum tham chiếu"
    root = REFERENCE_CAMPAIGN + "/"
    with ZipFile(bundle) as archive:
        assert len(archive.namelist()) == len(set(archive.namelist())), "Duplicate ZIP entries"
        assert archive.testzip() is None, "Reference ZIP CRC error"
        read = lambda name: json.loads(archive.read(root + name))
        frozen = read("campaign.json")
        base = __import__("yaml").safe_load(archive.read(root + "base_config.yaml"))
        search = __import__("yaml").safe_load(archive.read(root + "sweep.yaml"))
        assert content_hash(base) == frozen["base_config_sha256"]
        assert content_hash(search) == frozen["sweep_sha256"] and search["run_cap"] == 8
        assert read("sweep_id.json")["sweep_id"] == "upxyjxyl"
        prefix = "trials/" + WINNER_ID + "/"
        cfg = __import__("yaml").safe_load(archive.read(root + prefix + "resolved_config.yaml"))
        metrics = read(prefix + "metrics.json")
        trial = read(prefix + "sweep_trial.json")
        dataset = read(prefix + "dataset_identity.json")
        assert metrics["status"] == "completed" and metrics["best_epoch"] == 49
        assert metrics["best_val"]["mAP50-95"] == 0.4905282939017382
        assert content_hash(cfg) == metrics["config_sha256"] == trial["config_sha256"]
        assert content_hash(dataset) == trial["dataset_sha256"] == frozen["dataset_sha256"]
        assert trial["parameters"] == EXPECTED_WINNER
        assert frozen["nproc"] == 2 and frozen["test_policy"] == "shared"
        for key, value in EXPECTED_WINNER.items():
            section, field = key.split(".")
            assert cfg[section][field] == value
        return cfg, base, frozen, metrics

def make_ablation(winner, baseline, choice, data_root):
    cfg = copy.deepcopy(winner)
    section, field, value = CHOICES[choice]
    if choice in {"lr", "weight_decay", "dropout"}:
        assert baseline[section][field] == value
    elif choice == "jitter":
        assert baseline[section][field] == winner[section][field] == 0.1
    elif choice == "standard":
        assert winner[section][field] == "custom_mlp"
    cfg[section][field] = value
    cfg["train"].pop("run_name", None)
    cfg["data"]["root"] = str(Path(data_root).resolve())
    cfg["project"]["wandb_project"] = "sandbox"
    cfg = validate_config(cfg)
    assert cfg["model"]["pretrained"] and cfg["train"]["require_wandb"]
    assert cfg["seed"] == 42 and cfg["train"]["epochs"] == 100
    assert cfg["train"]["warmup_epochs"] == 3 and cfg["train"]["early_stopping_patience"] == 10
    assert cfg["train"]["batch_size"] == 24 and cfg["eval"]["batch_size"] == 32
    assert cfg["eval"]["auto_eval_test"] and cfg["online_preprocess"]["enabled"]
    return cfg

if REFERENCE_BUNDLE:
    bundle_path = Path(REFERENCE_BUNDLE)
else:
    candidates = sorted(INPUT_ROOT.rglob(REFERENCE_NAME))
    assert len(candidates) == 1, "Cần đúng 1 ZIP e2d80e82 + checksum; nếu nhiều thì điền REFERENCE_BUNDLE"
    bundle_path = candidates[0]
winner, baseline, frozen, winner_metrics = read_reference(bundle_path)
assert source_identity() == frozen["source_files"], "Source khác tuning; giữ pin a99d294"
assert runtime_identity() == frozen["runtime"], "Runtime đổi; gửi lỗi để review, không lách metadata"
DATA_ROOT = Path(load_yaml("configs/runtime/kaggle.yaml")["data_root_default"])
assert DATA_ROOT.is_dir(), f"Thiếu cache: {DATA_ROOT}"
cfg = make_ablation(winner, baseline, ABLATION, DATA_ROOT)
_, dataset = load_cache(cfg)
assert content_hash(dataset) == frozen["dataset_sha256"], "Cache khác tuning"
NAME = f"frcnn_ablation_{ABLATION}_s42_v1"
WORK = OUTPUT_ROOT / "frcnn_ablations" / NAME
request = dict(ablation=ABLATION, winner_run_id=WINNER_ID, reference_sha256=REFERENCE_SHA256,
               reference_sweep_id="upxyjxyl", config=cfg, dataset_sha256=frozen["dataset_sha256"],
               source_files=frozen["source_files"], runtime=frozen["runtime"],
               initialization="fresh COCO pretrained; winner checkpoint not loaded",
               selection="best validation mAP50-95; test report only")

# Khôi phục output của chính ablation này, không khôi phục tuning campaign.
def restore_output(bundle, target):
    bundle, target = Path(bundle), Path(target).resolve()
    assert not target.exists(), "Không ghi đè output đã có"
    assert file_hash(bundle) == bundle.with_suffix(".zip.sha256").read_text().split()[0]
    target.parent.mkdir(parents=True, exist_ok=True)
    staging = target.parent / (".restore_" + uuid.uuid4().hex)
    staging.mkdir()
    with ZipFile(bundle) as archive:
        assert len(archive.namelist()) == len(set(archive.namelist()))
        for entry in archive.infolist():
            dest = (staging / entry.filename).resolve()
            assert "\\" not in entry.filename and dest.is_relative_to(staging / target.name), "Unsafe path"
            assert not stat.S_ISLNK(entry.external_attr >> 16), "Symlink rejected"
        assert archive.testzip() is None
        archive.extractall(staging)
    restored = staging / target.name
    assert json.loads((restored / "request.json").read_text()) == request, "Sai ablation/config/source"
    restored.rename(target)

if RESTORE_OUTPUT and not WORK.exists():
    restore_output(RESTORE_OUTPUT, WORK)
if WORK.exists():
    assert json.loads((WORK / "request.json").read_text()) == request, "Output khác đối chứng; không ghi đè"
else:
    WORK.mkdir(parents=True)
    write_json(request, WORK / "request.json")
print("Đối chứng:", ABLATION, "— chỉ thay", CHOICES[ABLATION])
if ABLATION == "jitter":
    print("Jitter OFF, online master vẫn ON; đây không phải trả jitter về baseline (baseline=0.1).")
elif ABLATION == "standard":
    print("Standard head: head_hidden_dim/head_dropout giữ trong raw config nhưng không áp dụng.")
    print("So sánh thay head dưới recipe winner; chưa phải standard đã được tuning riêng.")
print("Output:", WORK)
print(json.dumps(cfg, indent=2))


## Một run độc lập — train mới, export kể cả khi controlled failure

Giữ START_TRAIN=True để Save & Run All chạy. Rerun cùng output completed sẽ kiểm và bỏ qua.
Run lỗi/dở không tự retry; giữ bundle để review. V1 không resume training.
Tên Forge: frcnn_ablation_<lr / weight_decay / dropout / jitter / standard>, mỗi run ID độc lập.


In [ ]:
from IPython.display import display, FileLink

def export_output(work):
    bundle = OUTPUT_ROOT / f"{work.name}-artifacts-{uuid.uuid4().hex[:8]}.zip"
    # W&B creates log/latest-run symlinks on Linux. Skip links instead of
    # dereferencing them; canonical regular files remain in the archive.
    skipped = sorted(p.relative_to(work).as_posix() for p in work.rglob("*") if p.is_symlink())
    files = sorted(p for p in work.rglob("*") if not p.is_symlink() and p.is_file()
                   and p.suffix.lower() not in {".zip", ".sha256"}
                   and p.name not in {".netrc", "_netrc", ".env"}
                   and p.relative_to(work).as_posix() != "export_receipt.json")
    with ZipFile(bundle, "x", compression=ZIP_STORED, allowZip64=True) as archive:
        for path in files:
            assert not path.is_symlink(), f"Output changed during export: {path.name}"
            archive.write(path, f"{work.name}/{path.relative_to(work).as_posix()}")
        archive.writestr(f"{work.name}/export_receipt.json", json.dumps(
            {"skipped_symlinks": skipped, "regular_files": len(files)}, indent=2))
    with ZipFile(bundle) as archive:
        assert archive.testzip() is None
    checksum = bundle.with_suffix(".zip.sha256")
    checksum.write_text(file_hash(bundle) + "  " + bundle.name + "\n", encoding="utf8")
    print("Tải ZIP + checksum ở Output gốc:", bundle, checksum, sep="\n")
    display(FileLink(str(bundle))); display(FileLink(str(checksum)))
    return bundle

def verify_completed(work):
    receipt = json.loads((work / "launch_status.json").read_text())
    assert receipt["status"] == "completed", "Run lỗi/dở: giữ output và review; không tự train lại"
    directory = work / "run"
    metrics = json.loads((directory / "metrics.json").read_text())
    saved = load_yaml(directory / "resolved_config.yaml")
    assert metrics["status"] == "completed" and metrics["config_sha256"] == content_hash(saved)
    expected = copy.deepcopy(cfg); expected["train"]["run_name"] = saved["train"]["run_name"]
    assert expected == saved
    assert content_hash(json.loads((directory / "dataset_identity.json").read_text())) == request["dataset_sha256"]
    assert file_hash(directory / "checkpoints/checkpoint_best.pt") == metrics["checkpoint_sha256"]
    tracking = json.loads((directory / "wandb_run.json").read_text())
    assert tracking["id"] == receipt["run_id"]
    score = metrics["best_val"]["mAP50-95"]
    assert isinstance(score, (int, float)) and math.isfinite(score)
    return metrics

# Worker được ghi ngoài repo pin; không sửa source training hoặc dùng checkpoint winner.
WORKER_CODE = r"""
import json, os, sys
from pathlib import Path
from vn_tsd.utils.io import load_yaml
from vn_tsd.pipelines.faster_rcnn.dataset import content_hash, load_cache
from vn_tsd.pipelines.faster_rcnn.sweep import source_identity, runtime_identity
from vn_tsd.pipelines.faster_rcnn.train import FasterRCNNPipeline
work = Path(sys.argv[1])
request = json.loads((work / 'request.json').read_text())
cfg = load_yaml(work / 'train_config.yaml')
expected = request['config']
comparison = json.loads(json.dumps(cfg)); comparison['train'].pop('run_name', None)
assert comparison == expected
assert source_identity() == request['source_files'] and runtime_identity() == request['runtime']
_, dataset = load_cache(cfg)
assert content_hash(dataset) == request['dataset_sha256']
assert not os.environ.get('WANDB_SWEEP_ID')
FasterRCNNPipeline(cfg, work / 'run').fit()
"""

def launch_ablation():
    marker = WORK / "launch_status.json"
    try:
        if marker.exists():
            metrics = verify_completed(WORK)
            print("Run đã hoàn tất; không train lại:", metrics["best_val"])
            return
        if not START_TRAIN:
            print("Chưa train. START_TRAIN=False")
            return
        assert not (WORK / "run").exists(), "Output dở không được ghi đè"
        run_id = uuid.uuid4().hex[:8]
        train_cfg = copy.deepcopy(cfg)
        train_cfg["train"]["run_name"] = NAME + "_" + run_id
        save_yaml(train_cfg, WORK / "train_config.yaml")
        (WORK / "worker.py").write_text(WORKER_CODE, encoding="utf8")
        command = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nnodes=1",
                   "--nproc_per_node=2", "--max_restarts=0", str(WORK / "worker.py"), str(WORK)]
        environment = dict(os.environ, WANDB_RUN_ID=run_id, WANDB_RESUME="never",
                           WANDB_PROJECT="sandbox", WANDB_DIR=str(WORK),
                           WANDB_RUN_GROUP="frcnn_ablation_s42_v1",
                           WANDB_TAGS=f"faster-rcnn,ablation,{ABLATION},seed42", PYTHONUNBUFFERED="1")
        for key in ("WANDB_SWEEP_ID", "WANDB_CONFIG_PATHS"):
            environment.pop(key, None)
        write_json(dict(status="running", run_id=run_id, command=command), marker)
        try:
            # Inherit stdout/stderr: Kaggle displays training progress directly.
            subprocess.run(command, cwd=REPO, env=environment, check=True)
            metrics = json.loads((WORK / "run/metrics.json").read_text())
            assert metrics["status"] == "completed"
            write_json(dict(status="completed", run_id=run_id, command=command), marker)
            verify_completed(WORK)
        except BaseException as exc:
            write_json(dict(status="interrupted" if isinstance(exc, KeyboardInterrupt) else "failed",
                            run_id=run_id, error_type=type(exc).__name__, command=command), marker)
            raise
    finally:
        export_output(WORK)

launch_ablation()


## Sau khi chạy

Trên Forge sandbox: các run thuộc group `frcnn_ablation_s42_v1`, tên chỉ rõ đối chứng.
Mỗi run có config/metrics/checkpoint độc lập. Không upload model Artifact tự động lên Forge.
Tải ZIP+SHA256 về `.agents/artifacts/notebooks/frcnn_ablation_<lựa_chọn>_s42_v1/`.
Gửi đủ năm output để review trước khi khóa config và chạy các seed còn lại.
Nếu Kaggle hard-kill thì finally không được bảo đảm chạy; giữ Save Version outputs để phục hồi/review.
